In [26]:
from pathlib import Path

from dotenv import load_dotenv
from loguru import logger

# Load environment variables from .env file if it exists
load_dotenv()

# Paths
PROJ_ROOT = "C:/users/10190/machine_learning/mle"
logger.info(f"PROJ_ROOT path is: {PROJ_ROOT}")

DATA_DIR = PROJ_ROOT + "/data"
RAW_DATA_DIR = DATA_DIR + "/raw"
INTERIM_DATA_DIR = DATA_DIR + "/interim"
PROCESSED_DATA_DIR = DATA_DIR + "/processed"
EXTERNAL_DATA_DIR = DATA_DIR + "/external"
MODELS_DIR = PROJ_ROOT + "/models"
REPORTS_DIR = PROJ_ROOT + "/reports"
FIGURES_DIR = REPORTS_DIR + "/figures"



2026-01-17 23:47:09.522 | INFO     | __main__:<module>:11 - PROJ_ROOT path is: C:/users/10190/machine_learning/mle


In [ ]:
from pathlib import Path
import pandas as pd
# import typer
import openpyxl 
import yaml



def load_data(input_path):

    df = pd.read_csv(input_path)
    print(f"Data loaded from {input_path} with shape {df.shape}")
    return df
   
    
def rename_columns(df, params):
    
    print(params["rename_columns"])

    df = df.rename(columns=params["rename_columns"])
    return df

def clean_data(df):
    df = df.dropna()
    
    df = df.drop_duplicates()
    
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # Convert withdrawal to numeric, fill NaN with 0
    df['withdrawal'] = pd.to_numeric(df['withdrawal'], errors='coerce').fillna(0)
    
    # Convert deposit to numeric, fill NaN with 0
    df['deposit'] = pd.to_numeric(df['deposit'], errors='coerce').fillna(0)
    
    # Convert balance to numeric, fill NaN with 0
    df['balance'] = pd.to_numeric(df['balance'], errors='coerce').fillna(0)
    
    # Reset index after cleaning
    df.reset_index(drop=True, inplace=True)
    return df

def prepare_data(df, Output_path):

    # customers_df = {}
    print("Preparing data for each customer...")

    # Ensure datetime
    df['date'] = pd.to_datetime(df['date'])

    unique_customers = df['customer_id'].unique()
    print(unique_customers)

    for customer in unique_customers:

        customer_data = df[df['customer_id'] == customer].copy()

        daily = (
            customer_data
            .groupby('date')
            .agg({
                'withdrawal': 'sum',
                'deposit': 'sum',
                'customer_id': 'count'   # number of rows = txn count
            })
            .rename(columns={'customer_id': 'txn_count'})
            .reset_index()
            .sort_values('date')
        )

        daily.to_csv(Output_path + f"/customer_{customer}_daily.csv", index=False)
    # return customers_df




def main(
    # ---- REPLACE DEFAULT PATHS AS APPROPRIATE ----
    input_path: Path = RAW_DATA_DIR + "/dataset.csv",
    output_path: Path = INTERIM_DATA_DIR,
    # ----------------------------------------------
):
    # ---- REPLACE THIS WITH YOUR OWN CODE ----

    home_dir = PROJ_ROOT
    print(f"Home directory is set to: {home_dir}")

    params_file = home_dir + "/params.yaml"
    params = yaml.safe_load(open(params_file))["dataset"]
    print(f"Parameters loaded: {params}")

    df = load_data(input_path)

    df = rename_columns(df, params)
    print("Columns renamed.")

    df = clean_data(df)

    prepare_data(df, output_path)

    # -----------------------------------------

main()

Home directory is set to: C:/users/10190/machine_learning/mle
Parameters loaded: {'test_size': 0.2, 'random_state': 42, 'rename_columns': {'customer_id': 'customer_id', 'date': 'date', 'transaction_type': 'transaction_type', 'description': 'description', 'deposit': 'deposit', 'withdrawal': 'withdrawal', 'balance': 'balance'}}
Data loaded from C:/users/10190/machine_learning/mle/data/raw/dataset.csv with shape (4781, 7)
{'customer_id': 'customer_id', 'date': 'date', 'transaction_type': 'transaction_type', 'description': 'description', 'deposit': 'deposit', 'withdrawal': 'withdrawal', 'balance': 'balance'}
Columns renamed.
Preparing data for each customer...
['CUST_1' 'CUST_2' 'CUST_3' 'CUST_4' 'CUST_5' 'CUST_6' 'CUST_7' 'CUST_8'
 'CUST_9' 'CUST_10']

Customer: CUST_1
        date  withdrawal  deposit  txn_count
0 2025-01-01        0.00  80357.0          2
1 2025-01-02    91359.00      0.0          4
2 2025-01-03    15328.00  19683.0          2
3 2025-01-04    10364.94      0.0          

In [5]:
current_directory = Path.cwd()
print(f"Current working directory: {current_directory}")

Current working directory: c:\users\10190\machine_learning\mle\notebooks
